# Notebook 3. Kernel statistics as the only covariates (MNIST)

**About this copy.** This is the MNIST copy of `model_fitting/3_kernel_only.ipynb` on the zain branch. The original fits modular addition runs. This copy reads the tables that `mnist_sweep.py assemble` writes to `mnist/sweep/`, through `mnist_settings.py`. The changes are these.
- The initial scale $s$ (column `init_scale`) takes the place of `alpha`.
- The grokking step `T` is the first step with test accuracy at least 0.80, where the original uses 0.95.
- The step budget is 500,000, and it is read from the data.
- The prediction time $t^*$ is `M.LANDMARK` from `mnist_settings.py`, where the original uses step 2,430.
- The original keeps weight decay up to 1e-3. This copy drops every weight decay level at which no run memorises.
- The held-out groups are the values of $s$ found in the data.
- All results of the original were removed. They will be written once the MNIST sweep has run.

**Question.** Can the kernel statistics `A_t`, `S_t` and `R_t`, read at a step before any run groks, tell us when a run will grok? Do they act alone or together, and what shape does the relation take?

Notebooks 1 and 2 also gave the models the settings ($s$, width and weight decay). If the 5 seeds of one cell behave almost the same, a model can learn each cell's typical grokking step and ignore the kernel. Here the models see only the kernel statistics, read at the prediction time $t^*$. Notebook 4 asks the same questions for runs that grok, and measures how much of the data is noise.

**The model with interactions.** Most of this notebook builds on one regression. With $z_A$, $z_S$, $z_R$ the standardised statistics (using log `R_t`, as in the modular addition notebook),
$$\log T = \beta_0 + \beta_A z_A + \beta_S z_S + \beta_R z_R + \gamma_{AS}\, z_A z_S + \gamma_{AR}\, z_A z_R + \gamma_{SR}\, z_S z_R + \sigma\varepsilon, \qquad \varepsilon \sim N(0, 1).$$
Each $\gamma$ term is an interaction term, the product of two standardised statistics. A non-zero $\gamma_{SR}$ means the slope of $\log T$ in $z_S$ changes by $\gamma_{SR}$ for each standard deviation of $z_R$, and the reverse. Runs that never grok enter as "$T$ is above 500,000".

**Takeaway.** To be written once the MNIST sweep has run.

## Words used in this notebook

| Term | Meaning |
| --- | --- |
| grokking step `T` | The first saved step at which test accuracy reaches 0.80. On MNIST this level replaces the 0.95 of golden report §3.5. |
| censored run | A run that never reaches 0.80 within 500,000 steps. We only know that `T` is above 500,000. These runs are kept. |
| cell | One mix of $s$, width and weight decay. Each cell has 5 seeds. |
| `S_t` | Kernel scale. The size of the kernel at step t divided by its size at the start (golden report Eq. 6). 1 means no change and below 1 means it shrank. |
| `R_t` | Kernel rotation. One minus the cosine between the kernel at step t and at the start. 0 means the same shape. |
| `A_t` | Kernel alignment. How well the kernel lines up with the labels, between 0 and 1. It uses the labels of a fixed draw of 1000 test images, the images on which the report kernel is computed. `T` uses all 10,000 test images. |
| prediction time $t^*$ | The kernel step nearest `M.LANDMARK_TARGET`, given by `M.LANDMARK`. Every run must still be ungrokked here, and the notebook checks this. The statistics are read at this step. Survival analysis calls it the landmark time. |
| cross-sectional covariates | The three statistics at $t^*$. Parts 1 to 4 use these. |
| longitudinal covariates | The statistics at all kernel steps up to $t^*$, a short time series for each run. Part 5 uses these. |
| standardised | Minus the mean and divided by the standard deviation of the training runs. We write $z_A$, $z_S$, $z_R$. |
| interaction term | The product of two standardised statistics, such as $z_A z_S$. It lets the effect of one statistic depend on the other. |
| AFT model | Accelerated failure time model. A regression of $\log T$ that counts a censored run as "$T$ is above 500,000". |
| concordance | Pick two runs. It is the share of pairs that the model puts in the right order. 0.5 is a coin toss and 1 is perfect. |
| validation set | Training runs held back to choose a model's settings. They are never the test runs. |

## How the models are tested

**The data inside each run.** Each network trains on 1000 MNIST training images, drawn by data seed 42 + seed, and is tested on all 10,000 test images. Test accuracy on those 10,000 images defines `T`. The report kernel statistics are computed on a fixed draw of 1000 test images, because a kernel on all 10,000 would have $10^8$ entries at every checkpoint (see the docstring of `mnist/mnist_features.py`).

**No random train/test split of runs.** The 5 seeds of a cell may be near copies. A random split would then put copies on both sides, and the test score would reward remembering cells. So we always hold out whole groups of runs.

**Outer loop: hold out one value of $s$.** There is one fold for each value of $s$ in the data. Each fold fits on the other values and predicts the held-out one. The table of folds is printed after the data is loaded.

The model never sees a run with the held-out $s$. We score each fold with the concordance and report the mean over the folds. Part 2 is the exception. It fits all runs to describe links and makes no claim about prediction.

**Inner loop: choosing model settings.** Settings are chosen from the training runs of each fold only.
- XGBoost holds out 20% of the training cells as a validation set. It tries 24 settings (tree depth 1 to 4, learning rate 0.03 or 0.1, minimum child weight 1, 5 or 10). Each one stops when the validation loss has not improved for 50 rounds. The setting with the best validation concordance is refitted on all training runs.
- Kernel ridge chooses its penalty and width by 5-fold cross-validation that holds out whole cells.
- The linear AFT has no settings to choose. A small fixed penalty of 0.01 keeps it stable.
- The GRU network in Part 5 has a fixed design. Its number of training epochs is chosen by early stopping on a validation set of cells.

**Repeats.** The seed changes the validation draw, the XGBoost subsampling and the starting weights of the GRU. We repeat these models over seeds and show the spread. The modular addition notebook treated a gap below about 0.03 in concordance as noise. Check that threshold against the seed spread on MNIST.

## Setup

In [ ]:
import copy
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb     # import XGBoost before PyTorch (Part 5), or their two copies of OpenMP clash
from scipy.stats import chi2
from lifelines import LogNormalAFTFitter
from lifelines.utils import concordance_index
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import GridSearchCV, GroupKFold

import mnist_settings as M

plt.style.use('fitting.mplstyle')
GREY, INK, PALE = '#8a8984', '#0b0b0b', '#c8c7c2'
KERNEL_COLOURS = {'A_t': '#7b5cd6', 'S_t': '#1a9e8f', 'R_t': '#c99a06', 'log R_t': '#c99a06'}
kernel = ['A_t', 'S_t', 'R_t']                                # the statistics as stored
covs = ['A_t', 'S_t', 'log R_t']                              # what the models use (log R_t, as in the modular addition notebook)
pairs = [('A_t', 'S_t'), ('A_t', 'R_t'), ('S_t', 'R_t')]
KNOB = M.KNOB                                                 # the initial scale s, column init_scale


def cindex(T, pred, grokked):
    # Concordance, or NaN when the group has no pair that can be ordered (for example when no run grokked).
    try:
        return concordance_index(T, pred, grokked)
    except ZeroDivisionError:
        return np.nan

## Load the data

`M.load()` reads the sweep tables and cuts the kernel table to about 48 log-spaced kernel steps, as in the original. We drop every weight decay level at which no run memorises (training accuracy at least `M.MEM`), since a network that never fits its training images cannot grok in the sense used here. `T` is the grokking step, or the step budget for a censored run.

In [ ]:
runs, acc, kern, LANDMARK = M.load()
T_STAR = LANDMARK                                             # prediction time t*
print('Data folder:', M.DATA)
print('Prediction time t* =', T_STAR, 'which is the kernel step nearest', M.LANDMARK_TARGET)

events = runs.set_index('run_id')
events['t_gen'] = acc[acc['test_acc'] >= M.LEVEL].groupby('run_id')['step'].min()
events['memorised'] = acc.groupby('run_id')['train_acc'].max() >= M.MEM
events = events.reset_index()
events['grokked'] = events['t_gen'].notna().astype(int)      # 1 = grokked, 0 = censored
events['T'] = events['t_gen'].fillna(events['last_step'])     # the grokking step, or the budget if censored
events['cell'] = events.apply(M.cell_name, axis=1)

memorises = events.groupby('eta_kappa')['memorised'].any()
dropped = memorises.index[~memorises].tolist()
print('Weight decay levels dropped because no run memorises:', dropped if dropped else 'none')
print('Weight decay levels kept:', memorises.index[memorises].tolist())

data = events.merge(kern[kern['step'] == T_STAR].drop(columns='step'), on='run_id')
data = data[data['eta_kappa'].isin(memorises.index[memorises])].reset_index(drop=True)
assert data['T'].min() > T_STAR, ('A run has grokked at or before the prediction time t*. '
                                  'Lower M.LANDMARK_TARGET in mnist_settings.py.')     # no run has grokked when we read the kernel
assert (data['R_t'] > 0).all(), 'R_t is 0 for some run at t*, so log R_t is undefined. Raise M.LANDMARK_TARGET.'
data['log R_t'] = np.log(data['R_t'])
scales = sorted(data[KNOB].unique())
print(f'{len(data)} runs, step budget {int(data["last_step"].max()):,}')
print('Folds:')
display(pd.DataFrame({'fit on': [', '.join(f'{v:g}' for v in scales if v != s) for s in scales],
                      'predict': [f'{s:g}' for s in scales]}, index=pd.Index(range(1, len(scales) + 1), name='fold')))
data.groupby(KNOB)['grokked'].agg(runs='size', grokked='sum')

## Part 1. Look at the data

**Question.** How does each statistic at $t^*$ relate to the grokking step?

Each dot is one run. Filled dots grokked, and their height is `T`. Hollow dots never grokked and are drawn at the step budget. `R_t` is drawn on a log axis, as in the modular addition notebook.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
grokked, censored = data[data['grokked'] == 1], data[data['grokked'] == 0]
for ax, stat in zip(axes, kernel):
    ax.scatter(grokked[stat], grokked['T'], s=12, color=KERNEL_COLOURS[stat], alpha=0.6, label='grokked')
    ax.scatter(censored[stat], censored['T'], s=12, facecolors='none', edgecolors=GREY, label='never grokked')
    ax.set_xlabel(stat)
    ax.set_yscale('log')
axes[2].set_xscale('log')
axes[2].set_xlabel('R_t (log scale)')
axes[0].set_ylabel('Grokking step T')
axes[0].legend(loc='lower left', fontsize=8)
plt.tight_layout()
plt.show()

Next, how the statistics differ between the values of $s$, and how strongly they move together. If one value of $s$ sits apart from the others, a model trained on the other values has to reach outside what it has seen.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, stat in zip(axes, kernel):
    groups = data.groupby(KNOB)
    box = ax.boxplot([g[stat] for _, g in groups], tick_labels=[f'{v:g}' for v, _ in groups], patch_artist=True, widths=0.5)
    for patch in box['boxes']:
        patch.set_facecolor(KERNEL_COLOURS[stat])
        patch.set_alpha(0.6)
    ax.set_xlabel('initial scale s')
    ax.set_title(stat)
axes[2].set_yscale('log')
plt.tight_layout()
plt.show()
print('Correlation between the covariates:')
data[covs].corr().round(2)

**Takeaway.** To be written once the MNIST sweep has run.

## Part 2. Which statistics are linked to grokking, and do they interact?

**Question.** Holding the others fixed, does each statistic go with earlier or later grokking? And does the effect of one depend on another?

We fit the AFT model with interactions from the top of the notebook on all runs. A coefficient $a$ multiplies the median grokking step by $e^a$ for one standard deviation more of that term. A negative $a$ means sooner. This part describes links in the data. Part 3 tests prediction.

We fit three versions, to see which findings depend on how the statistics are measured:
1. **raw `R_t`**: `R_t` on its own scale.
2. **log `R_t`**: our main version, as in the modular addition notebook. Part 1 shows whether the log suits `R_t` on MNIST.
3. **log `R_t` + squares**: version 2 plus $z_A^2$, $z_S^2$ and $z_R^2$. A square lets a statistic bend the curve on its own, so an interaction that only stood in for a bend should shrink here.

In versions 2 and 3, every term labelled `R_t` uses log `R_t`. The likelihood ratio (LR) test below asks whether the three interaction terms improve the fit in each version. It treats all runs as independent. If the 5 seeds of a cell are near copies, its p-values are too small. The cell bootstrap after it gives the fairer error bars.

In [ ]:
specs = ['raw R_t', 'log R_t', 'log R_t + squares']


def design(df, spec, interactions=True, ref=None):
    # Standardise the statistics with the mean and spread of `ref` (default: all runs), then add product and square terms.
    ref = data if ref is None else ref
    cols = ['A_t', 'S_t', 'R_t' if spec == 'raw R_t' else 'log R_t']
    z = (df[cols] - ref[cols].mean()) / ref[cols].std()
    z.columns = kernel     # in the log versions, the R_t column holds log R_t
    X = z.copy()
    if interactions:
        for a, b in pairs:
            X[f'{a} × {b}'] = z[a] * z[b]
    if spec == 'log R_t + squares':
        for c in kernel:
            X[f'{c}²'] = z[c] ** 2
    return X


def fit_aft(df, X, penalizer=1e-4):
    return LogNormalAFTFitter(penalizer=penalizer).fit(X.assign(T=df['T'].to_numpy(), grokked=df['grokked'].to_numpy()), 'T', 'grokked')


fits, lr_tests = {}, {}
for spec in specs:
    fits[spec] = fit_aft(data, design(data, spec))
    without = fit_aft(data, design(data, spec, interactions=False))
    lr = 2 * (fits[spec].log_likelihood_ - without.log_likelihood_)
    lr_tests[spec] = {'LR statistic': round(lr, 1), 'p': f'{chi2.sf(lr, 3):.1e}'}
pd.DataFrame(lr_tests).T

### How sure are we, and does it depend on the version?

We redo each fit 200 times. Each time we draw whole cells at random with replacement, so the 5 seeds of a cell stay together. The middle 95% of the refitted coefficients gives an error bar that respects the near copies. A finding we can trust keeps its sign, and its bar stays clear of 0, in all three versions.

In [ ]:
rng = np.random.default_rng(0)
cells = data['cell'].unique()
boot = []
for _ in range(200):
    pick = rng.choice(cells, size=len(cells), replace=True)
    sample = data.iloc[np.concatenate([np.flatnonzero(data['cell'] == c) for c in pick])].reset_index(drop=True)
    for spec in specs:
        boot.append({'spec': spec, **fit_aft(sample, design(sample, spec)).params_.loc['mu_'].drop('Intercept')})
boot = pd.DataFrame(boot)

terms = kernel + [f'{a} × {b}' for a, b in pairs]
fig, ax = plt.subplots(figsize=(8, 4))
for j, (spec, colour) in enumerate(zip(specs, [PALE, INK, KERNEL_COLOURS['R_t']])):
    estimate = fits[spec].params_.loc['mu_'][terms]
    spread = boot.loc[boot['spec'] == spec, terms]
    y = np.arange(len(terms)) + (j - 1) * 0.22
    ax.hlines(y, spread.quantile(0.025), spread.quantile(0.975), color=colour)
    ax.scatter(estimate, y, color=colour, zorder=3, label=spec)
ax.axvline(0, color=GREY, linestyle=':')
ax.set_yticks(range(len(terms)), terms)
ax.invert_yaxis()
ax.set_xlabel('Coefficient a (negative = groks sooner), line = 95% cell bootstrap')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

main = boot.loc[boot['spec'] == 'log R_t', terms]
pd.DataFrame({'coefficient': fits['log R_t'].params_.loc['mu_'][terms], 'lower 95%': main.quantile(0.025), 'upper 95%': main.quantile(0.975)}).round(2)

### What each interaction does

An interaction is easiest to read as a family of curves. Each panel slides one statistic from 2 standard deviations below its mean to 2 above. It does so at three values of a second statistic: one standard deviation below its mean, at its mean, and one above. The third statistic stays at its mean. The curves come from the main version (log `R_t`).

Parallel curves mean the pair does not interact. Curves that fan out or cross mean the effect of the first statistic depends on the second.

In [ ]:
shown = {'A_t': 'A_t', 'S_t': 'S_t', 'R_t': 'log R_t'}
gamma = fits['log R_t'].params_.loc['mu_']
grid = np.linspace(-2, 2, 50)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, (a, b) in zip(axes, pairs):
    for level, shade, label in zip([-1, 0, 1], [PALE, GREY, INK], ['-1 sd', 'mean', '+1 sd']):
        probe = pd.DataFrame(0.0, index=range(len(grid)), columns=kernel)
        probe[a], probe[b] = grid, level
        for p, q in pairs:
            probe[f'{p} × {q}'] = probe[p] * probe[q]
        ax.plot(grid, fits['log R_t'].predict_median(probe[list(design(data, 'log R_t'))]).to_numpy().ravel(), color=shade, label=f'{shown[b]} at {label}')
    ax.set_title(f'{shown[a]} × {shown[b]}:  γ = {gamma[f"{a} × {b}"]:+.2f}')
    ax.set_yscale('log')
    ax.set_xlabel(f'{shown[a]} (standard deviations from the mean)')
    ax.legend(loc='upper right', fontsize=8)
axes[0].set_ylabel('Predicted median grokking step')
plt.tight_layout()
plt.show()

**Takeaway.** To be written once the MNIST sweep has run.

## Part 3. Can the statistics rank runs from a value of $s$ the model has never seen?

**Question.** Fit on all values of $s$ but one and predict the held-out one. Does the model still put runs in the right order?

We climb a ladder of models. All of them use $z_A$, $z_S$ and $z_R$ with log `R_t`.
1. Each statistic on its own, in a linear AFT.
2. The linear AFT on all three, without and then with the interaction terms.
3. **Kernel ridge regression.** It fits a smooth curved surface. It predicts $\log T$, so it only learns from runs that grokked.
4. **XGBoost.** A sum of small decision trees, each one fitted to the errors of the trees before it. Its `survival:aft` objective uses the same likelihood as the AFT, so it keeps the censored runs.

As a reference we also fit **XGBoost on the settings only**: width and weight decay, without the kernel. $s$ is the held-out factor, so it cannot be used. If a kernel model does no better than this, the kernel adds nothing that the settings do not already give.

In [ ]:
pred = pd.DataFrame(index=data.index, columns=['A_t only', 'S_t only', 'R_t only', 'linear AFT', 'linear AFT + interactions', 'kernel ridge'], dtype=float)
for s in scales:
    train, test = (data[KNOB] != s).to_numpy(), (data[KNOB] == s).to_numpy()
    for name, interactions in [('linear AFT', False), ('linear AFT + interactions', True)]:
        X = design(data, 'log R_t', interactions, ref=data[train])
        pred.loc[test, name] = fit_aft(data[train], X[train], penalizer=0.01).predict_median(X[test]).to_numpy().ravel()
    X = design(data, 'log R_t', False, ref=data[train])
    for stat in kernel:
        aft = fit_aft(data[train], X.loc[train, [stat]], penalizer=0.01)
        pred.loc[test, f'{stat} only'] = aft.predict_median(X.loc[test, [stat]]).to_numpy().ravel()

Kernel ridge next, and the tools XGBoost needs: the validation split and the search over its settings.

In [ ]:
XGB_BASE = {'objective': 'survival:aft', 'aft_loss_distribution': 'normal', 'aft_loss_distribution_scale': 1.0,
            'subsample': 0.8, 'eval_metric': 'aft-nloglik'}
XGB_GRID = [{'max_depth': d, 'eta': e, 'min_child_weight': w} for d, e, w in itertools.product([1, 2, 3, 4], [0.03, 0.1], [1, 5, 10])]
log_T = np.log(data['T'])
upper = np.where(data['grokked'] == 1, data['T'], np.inf)     # a censored run's T can be anything above the budget


def validation_split(train, seed):
    # Hold out 20% of the training cells as a validation set. Returns the fitting rows and the validation rows.
    cells = data.loc[train, 'cell'].unique()
    held = np.random.default_rng(seed).choice(cells, round(0.2 * len(cells)), replace=False)
    val = train & data['cell'].isin(held).to_numpy()
    return train & ~val, val


def aft_matrix(X, rows):
    m = xgb.DMatrix(X[rows])
    m.set_float_info('label_lower_bound', data['T'].to_numpy()[rows])
    m.set_float_info('label_upper_bound', upper[rows])
    return m


def tuned_xgb(X, train, seed):
    # Try every setting on the validation set, keep the best, and refit it on all training runs.
    fit, val = validation_split(train, seed)
    best = (-1, None, None)
    for setting in XGB_GRID:
        params = {**XGB_BASE, **setting, 'seed': seed}
        booster = xgb.train(params, aft_matrix(X, fit), 1000, evals=[(aft_matrix(X, val), 'validation')],
                            early_stopping_rounds=50, verbose_eval=False)
        rounds = booster.best_iteration + 1
        c = cindex(data['T'].to_numpy()[val], booster.predict(aft_matrix(X, val), iteration_range=(0, rounds)),
                   data['grokked'].to_numpy()[val])
        if best[1] is None or c > best[0]:
            best = (c, params, rounds)
    return xgb.train(best[1], aft_matrix(X, train), best[2]), best


for s in scales:
    train, test = (data[KNOB] != s).to_numpy(), (data[KNOB] == s).to_numpy()
    X = design(data, 'log R_t', False, ref=data[train])
    g = train & (data['grokked'] == 1).to_numpy()
    krr = GridSearchCV(KernelRidge(kernel='rbf'), {'alpha': [0.01, 0.1, 1], 'gamma': [0.1, 0.5, 2]}, cv=GroupKFold(5))
    krr.fit(X[g], log_T[g], groups=data.loc[g, 'cell'])
    pred.loc[test, 'kernel ridge'] = np.exp(krr.predict(X[test]))

XGBoost runs with 5 seeds, on the kernel and on the settings only. The table shows the settings XGBoost chose with seed 0.

In [ ]:
# Trees only use the order, so where weight decay 0 sits does not matter, as long as it is below every positive level.
decay_floor = data.loc[data['eta_kappa'] > 0, 'eta_kappa'].min() / 10 if (data['eta_kappa'] > 0).any() else 1.0
settings_X = np.column_stack([np.log(data['width']), np.log10(data['eta_kappa'].where(data['eta_kappa'] > 0, decay_floor))])
fold_cols = [f's {s:g}' for s in scales]
xgb_scores, chosen = [], {}
for seed in range(5):
    for name in ['XGBoost', 'XGBoost, settings only']:
        p = np.zeros(len(data))
        for s in scales:
            train, test = (data[KNOB] != s).to_numpy(), (data[KNOB] == s).to_numpy()
            X = design(data, 'log R_t', False, ref=data[train]).to_numpy() if name == 'XGBoost' else settings_X
            booster, best = tuned_xgb(X, train, seed)
            p[test] = booster.predict(xgb.DMatrix(X[test]))
            if seed == 0 and name == 'XGBoost':
                chosen[f's {s:g} held out'] = {'depth': best[1]['max_depth'], 'learning rate': best[1]['eta'], 'min child weight': best[1]['min_child_weight'],
                                               'rounds': best[2], 'validation concordance': round(best[0], 3)}
        xgb_scores.append({'model': name, 'seed': seed, **{f's {s:g}': cindex(data.loc[data[KNOB] == s, 'T'], p[data[KNOB] == s], data.loc[data[KNOB] == s, 'grokked']) for s in scales}})
xgb_scores = pd.DataFrame(xgb_scores)
xgb_scores['mean'] = xgb_scores[fold_cols].mean(axis=1)
pd.DataFrame(chosen).T

### The scores

The bar is the mean concordance over the held-out values of $s$. The dots are the folds. For XGBoost the bar is the mean of 5 seeds and the line is the range over seeds. A fold in which no pair of runs can be ordered (for example, one where no run grokked) has no concordance and is left out of the mean.

In [ ]:
scores = pd.DataFrame({name: {f's {s:g}': cindex(data.loc[data[KNOB] == s, 'T'], pred.loc[data[KNOB] == s, name], data.loc[data[KNOB] == s, 'grokked'])
                              for s in scales} for name in pred}).T
for name, g in xgb_scores.groupby('model', sort=False):
    scores.loc[name] = g[fold_cols].mean()
scores['mean'] = scores[fold_cols].mean(axis=1)
seed_range = xgb_scores.groupby('model')['mean'].agg(['min', 'max'])

fig, ax = plt.subplots(figsize=(8, 4))
colours = [KERNEL_COLOURS['A_t'], KERNEL_COLOURS['S_t'], KERNEL_COLOURS['R_t'], GREY, GREY, GREY, INK, PALE]
ax.barh(scores.index, scores['mean'], color=colours)
for i, (name, row) in enumerate(scores.iterrows()):
    ax.scatter(row[fold_cols], [i] * len(fold_cols), color='white', edgecolors=INK, s=25, zorder=3)
    if name in seed_range.index:
        ax.hlines(i, seed_range.loc[name, 'min'], seed_range.loc[name, 'max'], color=KERNEL_COLOURS['R_t'], linewidth=3, zorder=4)
ax.axvline(0.5, color=GREY, linestyle=':')
ax.set_xlim(min(0.3, np.nanmin(scores[fold_cols].to_numpy()) - 0.02), max(0.9, np.nanmax(scores[fold_cols].to_numpy()) + 0.02))
ax.invert_yaxis()
ax.set_xlabel('Concordance on the held-out s (bar = mean, dots = each s)')
plt.tight_layout()
plt.show()
print(f'XGBoost range over 5 seeds:', seed_range.round(3).to_dict('index'))
scores.round(3)

**Takeaway.** To be written once the MNIST sweep has run.

## Part 4. What shape did XGBoost find?

**Question.** What does the fitted relation look like for each statistic?

We fit XGBoost once on all runs, choosing its settings on a validation set of 20% of the cells. Then we slide one statistic from low to high, hold the other two at their median, and plot the predicted grokking step. A flat line means the statistic has no effect at the median of the others. If the statistics are correlated, some points on a slice lie where there are no runs. Read the parts of each curve where most runs sit.

In [ ]:
X_all = design(data, 'log R_t', False)
booster, best = tuned_xgb(X_all.to_numpy(), np.ones(len(data), dtype=bool), seed=0)
print('chosen:', {k: best[1][k] for k in ['max_depth', 'eta', 'min_child_weight']}, 'rounds:', best[2])

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for ax, stat, col in zip(axes, kernel, covs):
    grid = np.linspace(X_all[stat].quantile(0.02), X_all[stat].quantile(0.98), 60)
    probe = pd.DataFrame({c: np.full(60, X_all[c].median()) for c in kernel})
    probe[stat] = grid
    x = grid * data[col].std() + data[col].mean()
    ax.plot(np.exp(x) if stat == 'R_t' else x, booster.predict(xgb.DMatrix(probe.to_numpy())), color=KERNEL_COLOURS[stat])
    ax.set_yscale('log')
    ax.set_xlabel(stat)
axes[2].set_xscale('log')
axes[0].set_ylabel('Predicted grokking step')
plt.tight_layout()
plt.show()

### Two statistics at once

This map holds `A_t` at its median and varies `S_t` and `R_t` together. Darker means a later predicted grokking step. The dots are the runs. Away from the dots the map is a guess.

In [ ]:
S, logR = np.meshgrid(np.linspace(data['S_t'].quantile(0.01), data['S_t'].quantile(0.99), 60),
                      np.linspace(data['log R_t'].quantile(0.01), data['log R_t'].quantile(0.99), 60))
probe = pd.DataFrame({'A_t': data['A_t'].median(), 'S_t': S.ravel(), 'log R_t': logR.ravel()})
surface = booster.predict(xgb.DMatrix(design(probe, 'log R_t', False).to_numpy())).reshape(S.shape)

fig, ax = plt.subplots(figsize=(6.4, 4.4))
im = ax.contourf(S, np.exp(logR), np.log10(surface), levels=12, cmap='Greys')
ax.scatter(data['S_t'], data['R_t'], s=6, color=KERNEL_COLOURS['S_t'], alpha=0.5)
fig.colorbar(im, label='log10 of predicted grokking step')
ax.set_yscale('log')
ax.set_xlabel('S_t')
ax.set_ylabel('R_t (log scale)')
ax.grid(False)
plt.tight_layout()
plt.show()

The steps in the curves come from tree splits, so small wiggles carry no meaning. Predicted steps above the 500,000 step budget lie beyond the data, and only their order matters.

**Takeaway.** To be written once the MNIST sweep has run.

## Part 5. Does the history of the kernel add information?

**Question.** Each run has the statistics at every kernel step up to $t^*$, a short time series. Is that series autocorrelated? And does it predict better than the last point alone?

First, what the series look like. Each line is one run, for up to 25 runs that grokked and 25 that never did.

In [ ]:
before = kern[kern['step'] <= T_STAR].pivot(index='run_id', columns='step', values=kernel)
history = np.stack([before[k].loc[data['run_id']].to_numpy() for k in kernel], axis=-1)     # runs × kernel steps up to t* × 3 statistics
steps = before['A_t'].columns.to_numpy()
assert not np.isnan(history).any(), 'Some run is missing a kernel step up to t*.'
print(f'{len(steps)} kernel steps up to t* = {T_STAR}, from {steps[0]} to {steps[-1]}')

rng = np.random.default_rng(1)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for i, (ax, stat) in enumerate(zip(axes, kernel)):
    for outcome, colour, label in [(1, KERNEL_COLOURS[stat], 'grokked'), (0, GREY, 'never grokked')]:
        candidates = np.flatnonzero(data['grokked'] == outcome)
        for k, r in enumerate(rng.choice(candidates, min(25, len(candidates)), replace=False)):
            ax.plot(steps[1:], history[r, 1:, i], color=colour, alpha=0.4, linewidth=0.8, label=label if k == 0 else None)
    ax.set_xscale('log')
    ax.set_xlabel('step')
    ax.set_title(stat)
axes[2].set_yscale('log')
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

Now the autocorrelation: the correlation between a series and itself one saved step earlier. The kernel steps are spaced on a log scale. The ratio between the last two kernel steps before $t^*$ is printed below. We measure the autocorrelation for the level and for the change between steps, and give the median over runs. The Ljung-Box test asks if the changes have any autocorrelation up to lag 3. The last column is the share of runs where it says yes at the 5% level.

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import acf

print(f'Near t*, the previous kernel step is {steps[-1] / steps[-2]:.2f} times earlier.')
rows = {}
for i, stat in enumerate(kernel):
    level, change = history[:, :, i], np.diff(history[:, :, i], axis=1)
    rows[stat] = {'lag-1 autocorrelation of the level': np.median([acf(r, nlags=1)[1] for r in level]),
                  'lag-1 autocorrelation of the change': np.median([acf(r, nlags=1)[1] for r in change]),
                  'share of runs with Ljung-Box p < 0.05': np.mean([acorr_ljungbox(r, lags=[3])['lb_pvalue'].iloc[0] < 0.05 for r in change])}
pd.DataFrame(rows).T.round(3)

Autocorrelation shows structure over time, but it does not show that the structure predicts `T`. The models below test that.

**Takeaway.** To be written once the MNIST sweep has run.

### Models of the history

We use the hold-one-$s$-out test of Part 3 with 3 seeds and compare three models.
- **XGBoost, cross-sectional.** The XGBoost of Part 3, which sees the statistics at $t^*$ only. Its scores are seeds 0 to 2 from Part 3.
- **XGBoost, longitudinal.** XGBoost on all values of the history (kernel steps × 3 statistics) as one row. Trees ignore the time order.
- **GRU, longitudinal.** A gated recurrent unit network with 16 hidden units. It reads the history one step at a time, in time order, and predicts $\log T$ from its final state. It is trained with the AFT likelihood, so censored runs still count as "above 500,000". Its learning rate (0.01) and weight decay ($10^{-4}$) are fixed. Early stopping chooses the number of epochs: we fit on 80% of the training cells, check the loss on the other 20% every 10 epochs up to 800, and keep the best weights.

PyTorch runs on one thread here, because PyTorch and XGBoost each bring their own copy of OpenMP and can otherwise hang the kernel.

In [ ]:
import torch
torch.set_num_threads(1)


def aft_loss(mu, log_sigma, y, grokked):
    # Log-normal AFT loss: the density for runs that grokked, the chance of a later step for runs that did not.
    z = (y - mu) / log_sigma.exp()
    normal = torch.distributions.Normal(0.0, 1.0)
    return -(grokked * (normal.log_prob(z) - log_sigma) + (1 - grokked) * torch.log(normal.cdf(-z) + 1e-12)).mean()


class GRU(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = torch.nn.GRU(3, 16, batch_first=True)
        self.head = torch.nn.Linear(16, 1)
        self.log_sigma = torch.nn.Parameter(torch.zeros(()))

    def forward(self, x):
        return self.head(self.gru(x)[1][-1]).squeeze(-1)


def train_gru(z, train, seed):
    # Fit on 80% of the training cells and keep the weights with the lowest loss on the other 20%.
    fit, val = validation_split(train, seed)
    y = (log_T - log_T[train].mean()) / log_T[train].std()
    Z, Y, E = (torch.tensor(np.asarray(v), dtype=torch.float32) for v in (z, y, data['grokked']))
    torch.manual_seed(seed)
    net = GRU()
    opt = torch.optim.Adam(net.parameters(), lr=1e-2, weight_decay=1e-4)
    best = (np.inf, None, 0)
    for epoch in range(1, 801):
        opt.zero_grad()
        aft_loss(net(Z[fit]), net.log_sigma, Y[fit], E[fit]).backward()
        opt.step()
        if epoch % 10 == 0:
            with torch.no_grad():
                loss = aft_loss(net(Z[val]), net.log_sigma, Y[val], E[val]).item()
            if loss < best[0]:
                best = (loss, copy.deepcopy(net.state_dict()), epoch)
    net.load_state_dict(best[1])
    return net, best[2]


hist_scores, epochs = [], {}
for seed in range(3):
    preds = {name: np.zeros(len(data)) for name in ['XGBoost, longitudinal', 'GRU, longitudinal']}
    for s in scales:
        train, test = (data[KNOB] != s).to_numpy(), (data[KNOB] == s).to_numpy()
        z = (history - history[train].mean(axis=(0, 1))) / history[train].std(axis=(0, 1))
        booster, _ = tuned_xgb(z.reshape(len(z), -1), train, seed)
        preds['XGBoost, longitudinal'][test] = booster.predict(xgb.DMatrix(z.reshape(len(z), -1)[test]))
        net, epochs[seed, s] = train_gru(z, train, seed)
        with torch.no_grad():
            preds['GRU, longitudinal'][test] = net(torch.tensor(z[test], dtype=torch.float32)).numpy()
    for name, p in preds.items():
        hist_scores.append({'model': name, 'seed': seed, **{f's {s:g}': cindex(data.loc[data[KNOB] == s, 'T'], p[data[KNOB] == s], data.loc[data[KNOB] == s, 'grokked']) for s in scales}})

cross = xgb_scores[(xgb_scores['model'] == 'XGBoost') & (xgb_scores['seed'] < 3)].assign(model='XGBoost, cross-sectional')
compare = pd.concat([cross, pd.DataFrame(hist_scores)], ignore_index=True)
compare['mean'] = compare[fold_cols].mean(axis=1)
summary = compare.groupby('model', sort=False)[fold_cols + ['mean']].mean()
summary['lowest seed'] = compare.groupby('model', sort=False)['mean'].min()
summary['highest seed'] = compare.groupby('model', sort=False)['mean'].max()
print('GRU epochs chosen by early stopping:', sorted(epochs.values()))

fig, ax = plt.subplots(figsize=(8, 2.4))
ax.barh(summary.index, summary['mean'], color=[INK, INK, KERNEL_COLOURS['R_t']])
ax.hlines(range(len(summary)), summary['lowest seed'], summary['highest seed'], color=GREY, linewidth=3)
ax.axvline(0.5, color=GREY, linestyle=':')
ax.set_xlim(min(0.4, summary['lowest seed'].min() - 0.02), max(0.85, summary['highest seed'].max() + 0.02))
ax.invert_yaxis()
ax.set_xlabel('Concordance on the held-out s (bar = mean of 3 seeds, line = seed range)')
plt.tight_layout()
plt.show()
summary.round(3)

**Takeaway.** To be written once the MNIST sweep has run.

## Summary

The questions of this notebook are these.
- Are the statistics linked to grokking time?
- On what scale does `R_t` act?
- Do the statistics interact?
- What shape is the relation?
- Can the statistics rank runs from a new value of $s$?
- Does the history of the statistics help?
- Do the statistics beat the settings?

**Takeaway.** To be written once the MNIST sweep has run.

## How to model these data

**Takeaway.** To be written once the MNIST sweep has run.

## Conclusion

**Takeaway.** To be written once the MNIST sweep has run.